In [96]:
from pathlib import Path
import sys
import os
import importlib


# Loading python model directory as module into jupyter notebook
module_dir = Path(os.getcwd()).parent.resolve()
data_dir = module_dir / "data"

if module_dir not in sys.path:
    sys.path.insert(1, str(module_dir))
else:
    print("Module path already inserted into system paths")

try:
    from model import markov
    from model import constants

    # to apply changes in modules
    importlib.reload(markov)
    importlib.reload(constants)
except ModuleNotFoundError as e:
    print(f"Unable to import module: {e.msg}")

In [ ]:
# Model inputs
n_samples = 128
short_steps = constants.SHORT_TERM_CYCLE_COUNTS
long_steps = constants.LONG_TERM_CYCLE_COUNTS
# Loading states name from excel sheet is deprecated, now on only generate within the code blocks
start_state = "Healthy"
primary_states = [
    "Healthy",
    "Bleeding",
    "Hemarthrosis",
    "Arthropathy",
    "LT_Bleeding",
    "Death",
]
secondary_states = ["Healthy", "Bleeding", "Hemarthrosis", "LT_Bleeding", "Death"]
# NOTE:
# Transition matrix is dynamically generated through the *_psa functions
# To change in states need to update the psa worker functions as well to support new model schema
# NOTE:
# Newly suggested model structure:               switch
#                   [Healthy]                    ------>                     [Arthropathy]
#        |              |              |                           |              |              |
# [LT Bleeding] | [Hemarthrosis] | [Bleeding]               [LT Bleeding] | [Hemarthrosis] | [Bleeding]
#    |                  |                                          |
# [DEATH]         [Arthropathy]                                 [DEATH]

chains = {"primary": (primary_states, {}), "secondary": (secondary_states, {})}


# Define switch conditions
def arthropathy_switch_condition(step: int, state: str, chain: str, **kwargs) -> bool:
    """Determine if a switch to the secondary chain should occur based on the Arthropathy state."""
    return state == "Arthropathy" and chain == "primary"


switch_conditions = {"secondary": arthropathy_switch_condition}

# Short term simulation
on_demand_inputs, on_demand_outputs = markov.on_demand_psa(
    n_samples=n_samples,
    chains=chains,
    start_chain="primary",
    start_state="Healthy",
    steps=short_steps,
    switch_conditions=switch_conditions,
)

np.float64(20681.3203125)